# Train a Sentiment Classifier (DistilBERT on IMDB)

**How to use this notebook:**
1. Go to `Runtime` -> `Change runtime type` -> select **T4 GPU** -> Save.
2. Run each cell top to bottom (Shift+Enter).
3. Training takes ~10-20 minutes on the free GPU for a solid accuracy result.
4. At the end, you'll download a `model.zip` file. You'll need it for the backend step.

## 1. Install libraries

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

## 2. Load the dataset

We're using the IMDB dataset: 50,000 movie reviews labeled `positive` or `negative`. It ships pre-cleaned inside the `datasets` library, so there's no manual scraping or labeling needed.

To keep training fast today, we'll use a subset (5,000 train / 1,000 test). You can raise these numbers later for higher accuracy at the cost of more training time.

In [ ]:
from datasets import load_dataset

raw = load_dataset("imdb")

# Shuffle then take a manageable subset for a same-day training run
train_ds = raw["train"].shuffle(seed=42).select(range(5000))
test_ds = raw["test"].shuffle(seed=42).select(range(1000))

print(train_ds)
print(train_ds[0])

## 3. Tokenize the text

Models don't read words directly -- text gets converted into numeric tokens first. We use DistilBERT's own tokenizer so the numbers match what the model expects.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

train_tok = train_ds.map(tokenize, batched=True)
test_tok = test_ds.map(tokenize, batched=True)

train_tok = train_tok.rename_column("label", "labels")
test_tok = test_tok.rename_column("label", "labels")

train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

## 4. Load the pretrained model

DistilBERT already understands English from pretraining on huge amounts of text. We're not starting from scratch -- we're *fine-tuning* it: adjusting its knowledge slightly so it's good at our specific task (positive vs. negative). This is why we get strong accuracy without needing millions of examples.

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

## 5. Set up training

In [ ]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"],
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

## 6. Train

This is the step that actually adjusts the model's internal weights based on your data. Watch the `eval_f1` and `eval_accuracy` numbers printed after each epoch -- that's how you know it's actually learning.

In [ ]:
trainer.train()

## 7. Evaluate

Final accuracy on held-out test data the model never saw during training -- this is the honest measure of how good it is.

In [ ]:
trainer.evaluate()

## 8. Save the model and download it

This zips up everything the backend will need: the trained weights and the tokenizer.

In [ ]:
SAVE_DIR = "sentiment_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import shutil
shutil.make_archive("model", "zip", SAVE_DIR)

from google.colab import files
files.download("model.zip")

## 9. Quick sanity check

Try your own sentences before moving on.

In [ ]:
from transformers import pipeline

clf = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR)

print(clf("This movie completely blew me away, I loved every second of it."))
print(clf("What a waste of time, I want my two hours back."))
print(clf("It was fine. Nothing special, nothing terrible."))

## Optional: push to the Hugging Face Hub

Instead of downloading a zip and re-uploading it later, you can host the model directly on Hugging Face and load it by name in your backend. Requires a free account at huggingface.co and an access token (Settings -> Access Tokens).

In [ ]:
# from huggingface_hub import login
# login()  # paste your token when prompted
#
# trainer.push_to_hub("your-username/sentiment-distilbert")
# tokenizer.push_to_hub("your-username/sentiment-distilbert")